In [ ]:
#Qué hace: transforma eventos clínicos y de tratamiento en variables diarias por paciente.
#Clave: cambio de unidad de análisis → día del paciente.

In [1]:
# ============================================================
# 06_variables_diarias.ipynb  (VERSIÓN FINAL COMPLETA)
# ------------------------------------------------------------
# Objetivo:
# - A partir de ventanas diarias (df_windows) subirlas a BigQuery (tmp_table)
# - Extraer vitales de chartevents dentro de cada ventana:
#     HR, RR, SpO2, FiO2, MAP (arterial + NIBP), Temp (multi-item, F->C)
# - Agregar por día (mediana)
# - Normalizar FiO2 a fracción
# - Calcular S/F ratio
#
# Salida:
# - 06_variables_diarias.parquet
#
# NOTAS IMPORTANTES:
# - MIMIC-IV chartevents.charttime es DATETIME (no TIMESTAMP)
# - La tabla temporal (tmp_table) debe estar en US
# - Recomendación: dataset scratch propio (p.ej. mimic-pruebas.scratch)
# ============================================================

In [2]:
import pandas as pd
import numpy as np
from google.cloud import bigquery

In [3]:
# -----------------------------
# 0) Configuración
# -----------------------------
PROJECT_ID = "mimic-pruebas"
ICU = "physionet-data.mimiciv_3_1_icu"

# Dataset donde guardar la tabla temporal (debe existir en tu proyecto)
DATASET_TMP = "scratch"
TMP_TABLE_NAME = "tmp_06_stays_windows"

client = bigquery.Client(project=PROJECT_ID)
tmp_table = f"{PROJECT_ID}.{DATASET_TMP}.{TMP_TABLE_NAME}"

print("PROJECT_ID:", PROJECT_ID)
print("ICU:", ICU)
print("tmp_table:", tmp_table)

PROJECT_ID: mimic-pruebas
ICU: physionet-data.mimiciv_3_1_icu
tmp_table: mimic-pruebas.scratch.tmp_06_stays_windows


In [4]:
# -----------------------------
# 1) Cargar ventanas (df_windows)
# -----------------------------
df_windows = pd.read_parquet("05_ventanas_24h.parquet")

required_cols = {"icu_stay_id","subject_id","hadm_id","day_idx","window_start","window_end"}
missing = required_cols - set(df_windows.columns)
if missing:
    raise ValueError(f"Missing columns in df_windows: {missing}")

df_windows = df_windows.copy()

# Tipos
df_windows["icu_stay_id"] = df_windows["icu_stay_id"].astype("Int64")
df_windows["subject_id"]  = df_windows["subject_id"].astype("Int64")
df_windows["hadm_id"]     = df_windows["hadm_id"].astype("Int64")
df_windows["day_idx"]     = df_windows["day_idx"].astype(int)

df_windows["window_start"] = pd.to_datetime(df_windows["window_start"]).astype("datetime64[ns]")
df_windows["window_end"]   = pd.to_datetime(df_windows["window_end"]).astype("datetime64[ns]")

# Limpiar filas sin stay_id (no se pueden unir a chartevents)
df_windows = df_windows.dropna(subset=["icu_stay_id"]).copy()
df_windows["icu_stay_id"] = df_windows["icu_stay_id"].astype("int64")

global_start = df_windows["window_start"].min()
global_end   = df_windows["window_end"].max()

print("Global window range:", global_start, "->", global_end)
print("N windows:", len(df_windows), "| N stays:", df_windows["icu_stay_id"].nunique())

Global window range: 2110-01-13 20:00:00 -> 2214-07-27 16:00:00
N windows: 1130610 | N stays: 21080


In [5]:
# -----------------------------
# 2) Subir ventanas a BigQuery (tabla temporal)
# -----------------------------
# OJO: si tu dataset scratch no existe, créalo en BigQuery UI en location US.
df_win_small = df_windows[["icu_stay_id","subject_id","hadm_id","day_idx","window_start","window_end"]].copy()

job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
load_job = client.load_table_from_dataframe(df_win_small, tmp_table, job_config=job_config)
load_job.result()

print("Uploaded temp table:", tmp_table)

# Verificación rápida
n_tmp = client.query(f"SELECT COUNT(*) AS n FROM `{tmp_table}`", location="US").to_dataframe()
print(n_tmp)

Uploaded temp table: mimic-pruebas.scratch.tmp_06_stays_windows


E0000 00:00:1769593982.686789  948234 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


         n
0  1130610


In [6]:
# -----------------------------
# 3) ITEMIDs + mapping (Temp multi-fuente)
# -----------------------------
# Temp en MIMIC no es un itemid único: usamos varios y convertimos F->C.
TEMP_F_ITEMIDS = [223761]  # Temperature Fahrenheit
TEMP_C_ITEMIDS = [223762, 223763, 224027, 224028]  # Celsius / otras fuentes

ITEMIDS = [
    220045,  # HR
    220210,  # RR
    220277,  # SpO2
    223835,  # FiO2
    220052,  # MAP arterial
    220181,  # MAP NIBP mean
] + TEMP_F_ITEMIDS + TEMP_C_ITEMIDS

ITEM_TO_VAR = {
    220045: "HR",
    220210: "RR",
    220277: "SpO2",
    223835: "FiO2",
    220052: "MAP",
    220181: "MAP",
    223761: "Temp",
    223762: "Temp",
    223763: "Temp",
    224027: "Temp",
    224028: "Temp",
}

itemids_sql = ", ".join(map(str, ITEMIDS))

In [7]:
# -----------------------------
# 4) Query BigQuery: extraer chartevents dentro de ventanas
# -----------------------------
# IMPORTANTE: charttime es DATETIME y window_start/end también (por cómo sube pandas).
sql = f"""
WITH w AS (
  SELECT
    icu_stay_id,
    subject_id,
    hadm_id,
    day_idx,
    window_start,
    window_end
  FROM `{tmp_table}`
),
ce AS (
  SELECT
    ce.stay_id AS icu_stay_id,
    ce.charttime,   -- DATETIME
    ce.itemid,
    ce.valuenum
  FROM `{ICU}.chartevents` ce
  WHERE ce.itemid IN ({itemids_sql})
    AND ce.valuenum IS NOT NULL
    AND ce.charttime >= DATETIME('{global_start}')
    AND ce.charttime <  DATETIME('{global_end}')
)
SELECT
  w.subject_id,
  w.hadm_id,
  w.icu_stay_id,
  w.day_idx,
  ce.itemid,
  ce.valuenum
FROM w
JOIN ce
  ON ce.icu_stay_id = w.icu_stay_id
 AND ce.charttime >= w.window_start
 AND ce.charttime <  w.window_end
"""

df_long = client.query(sql, location="US").to_dataframe()

print("df_long shape:", df_long.shape)
print("Top itemids:\n", df_long["itemid"].value_counts().head(15))
display(df_long.head())

E0000 00:00:1769593999.218755  948234 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


df_long shape: (75963826, 6)
Top itemids:
 itemid
220045    17049356
220210    16922691
220277    16847169
220181     8445188
220052     8192039
223761     4226524
223835     3435222
223762      816674
223763       28963
Name: count, dtype: Int64


,subject_id,hadm_id,icu_stay_id,day_idx,itemid,valuenum
0,10194974,28046822,30046365,2,220045,41.0
1,16201174,20862682,35025325,1,220045,46.0
2,17689755,21438160,38033217,19,220045,47.0
3,17727739,23492347,37222849,2,220045,49.0
4,13215261,28394530,37527431,3,220045,49.0


In [8]:
# -----------------------------
# 5) Normalizaciones pre-agregación
# -----------------------------
df_long["itemid"] = df_long["itemid"].astype(int)
df_long["variable"] = df_long["itemid"].map(ITEM_TO_VAR)

if df_long["variable"].isna().any():
    missing_ids = df_long.loc[df_long["variable"].isna(), "itemid"].unique()
    raise ValueError(f"Unmapped itemids found: {missing_ids}")

# Fahrenheit -> Celsius
mask_temp_f = df_long["itemid"].isin(TEMP_F_ITEMIDS)
df_long.loc[mask_temp_f, "valuenum"] = (df_long.loc[mask_temp_f, "valuenum"] - 32.0) * 5.0 / 9.0

# Limpieza conservadora (evita outliers absurdos)
def apply_plausibility_filters(df):
    df = df.copy()

    # Temp (C)
    m = df["variable"] == "Temp"
    df.loc[m, "valuenum"] = df.loc[m, "valuenum"].where(df.loc[m, "valuenum"].between(25, 45))

    # HR
    m = df["variable"] == "HR"
    df.loc[m, "valuenum"] = df.loc[m, "valuenum"].where(df.loc[m, "valuenum"].between(10, 250))

    # RR
    m = df["variable"] == "RR"
    df.loc[m, "valuenum"] = df.loc[m, "valuenum"].where(df.loc[m, "valuenum"].between(2, 80))

    # MAP
    m = df["variable"] == "MAP"
    df.loc[m, "valuenum"] = df.loc[m, "valuenum"].where(df.loc[m, "valuenum"].between(20, 200))

    # SpO2
    m = df["variable"] == "SpO2"
    df.loc[m, "valuenum"] = df.loc[m, "valuenum"].where(df.loc[m, "valuenum"].between(0, 100))

    # FiO2 (positivo; normalización luego)
    m = df["variable"] == "FiO2"
    df.loc[m, "valuenum"] = df.loc[m, "valuenum"].where(df.loc[m, "valuenum"] > 0)

    return df

df_long = apply_plausibility_filters(df_long)

In [9]:
# -----------------------------
# 6) Agregar mediana diaria + pivot a formato ancho
# -----------------------------
df_agg = (
    df_long
    .groupby(["subject_id","hadm_id","icu_stay_id","day_idx","variable"], as_index=False)["valuenum"]
    .median()
)

df_daily = (
    df_agg
    .pivot_table(
        index=["subject_id","hadm_id","icu_stay_id","day_idx"],
        columns="variable",
        values="valuenum",
        aggfunc="first"
    )
    .reset_index()
)

df_daily = df_daily.sort_values(["subject_id","hadm_id","icu_stay_id","day_idx"]).reset_index(drop=True)

In [10]:
# -----------------------------
# 7) Normalizar FiO2 y calcular S/F ratio
# -----------------------------
def normalize_fio2(x):
    """
    Convert FiO2 to fraction.
    - If 21..100 -> percent
    - If 0..1.5  -> fraction
    - else -> NaN
    """
    if pd.isna(x):
        return np.nan
    x = float(x)
    if 1.5 < x <= 100:
        return x / 100.0
    if 0 < x <= 1.5:
        return x
    return np.nan

df_daily["FiO2_frac"] = df_daily["FiO2"].apply(normalize_fio2)
df_daily["SF_ratio"] = df_daily["SpO2"] / df_daily["FiO2_frac"]

In [11]:
# -----------------------------
# 8) QA final
# -----------------------------
cols_check = ["FiO2","Temp","MAP","HR","RR","SpO2"]
for c in cols_check:
    if c in df_daily.columns:
        print(f"{c} NaN rate:", df_daily[c].isna().mean())
    else:
        print(f"{c} MISSING COLUMN")

print("FiO2_frac NaN rate:", df_daily["FiO2_frac"].isna().mean())
print("SF_ratio NaN rate:", df_daily["SF_ratio"].isna().mean())

print("\nTemp summary (C):")
print(df_daily["Temp"].describe(percentiles=[.01,.05,.5,.95,.99]))


FiO2 NaN rate: 0.31575585017864044
Temp NaN rate: 0.0373922439289855
MAP NaN rate: 0.028740210267290265
HR NaN rate: 0.0009543264111806043
RR NaN rate: 0.003186346034024497
SpO2 NaN rate: 0.004046028503600413
FiO2_frac NaN rate: 0.31575585017864044
SF_ratio NaN rate: 0.3168679164924955

Temp summary (C):
count    122050.000000
mean         36.994608
std           0.563206
min          26.000000
1%           35.638889
5%           36.166667
50%          36.944444
95%          38.000000
99%          38.500000
max          42.100000
Name: Temp, dtype: float64


In [12]:
# -----------------------------
# 9) Guardar
# -----------------------------
out_path = "06_variables_diarias.parquet"
df_daily.to_parquet(out_path, index=False)
print("Saved:", out_path)

display(df_daily.head(10))

Saved: 06_variables_diarias.parquet


variable,subject_id,hadm_id,icu_stay_id,day_idx,FiO2,HR,MAP,RR,SpO2,Temp,FiO2_frac,SF_ratio
0,10001217,24597018,37067082,0,NaN,91.5,90.0,21.0,98.0,36.972222,NaN,NaN
1,10001217,24597018,37067082,1,NaN,98.0,94.0,23.0,95.0,37.611111,NaN,NaN
2,10002428,20321825,34807493,0,25.0,101.5,63.0,23.0,100.0,36.944444,0.25,400.0
3,10002428,20321825,34807493,1,NaN,102.5,67.5,22.5,97.0,37.027778,NaN,NaN
4,10002428,23473524,35479615,0,NaN,88.5,82.0,20.5,99.0,36.666667,NaN,NaN
5,10002428,23473524,35479615,1,NaN,89.0,81.5,20.5,99.0,36.694444,NaN,NaN
6,10002428,23473524,35479615,2,NaN,96.5,91.0,22.0,99.0,36.944444,NaN,NaN
7,10002428,28662225,33987268,0,NaN,97.0,68.0,18.0,100.0,36.583333,NaN,NaN
8,10002428,28662225,33987268,1,NaN,97.0,68.5,19.0,98.0,36.583333,NaN,NaN
9,10002428,28662225,33987268,2,NaN,96.0,64.5,19.5,96.5,36.555556,NaN,NaN
